In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Arc, FancyArrowPatch, Wedge
import numpy as np
import json
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

In [2]:
# fig09_isoscore_contour.png - Iso-score elliptical contours (Paper Fig.2)

alpha = 0.4

A = np.linspace(0, 1.1, 400)
B = np.linspace(0, 1.1, 400)
AA, BB = np.meshgrid(A, B)
S = np.sqrt(AA**2 + (alpha * BB)**2)

fig, ax = plt.subplots(figsize=(10, 9))

levels = [0.25, 0.50, 0.75, 1.00]
contour_colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52']
CS = ax.contour(AA, BB, S, levels=levels, colors=contour_colors, linewidths=2)
ax.clabel(CS, inline=True, fontsize=11, fmt={l: f's = {l:.2f}' for l in levels})

# Sample points
points = [
    {'label': 'Re-only', 'xy': (0.92, 0.22), 'marker': '^', 'color': 'red'},
    {'label': 'Im-only', 'xy': (0.05, 0.95), 'marker': 's', 'color': 'blue'},
    {'label': 'Balanced', 'xy': (0.70, 0.60), 'marker': 'o', 'color': 'green'},
]

for pt in points:
    a_val, b_val = pt['xy']
    score = np.sqrt(a_val**2 + (alpha * b_val)**2)
    ax.scatter(*pt['xy'], marker=pt['marker'], color=pt['color'], s=120, zorder=5)
    ax.annotate(
        f"{pt['label']}\ns={score:.2f}",
        xy=pt['xy'],
        xytext=(pt['xy'][0] + 0.08, pt['xy'][1] + 0.07),
        fontsize=10,
        color=pt['color'],
        arrowprops=dict(arrowstyle='->', color=pt['color'], lw=1.5),
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7)
    )

ax.text(0.55, 1.02, 'alpha=0.4 → Re 편향 soft-OR', fontsize=11,
        color='dimgray', ha='center', style='italic')
ax.text(0.03, 0.78, 'Im 단독 고점수 불가 → Re가 우세한 검색',
        fontsize=10, color='navy',
        bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow', ec='navy', alpha=0.85))

ax.set_xlim(0, 1.1)
ax.set_ylim(0, 1.1)
ax.set_xlabel('A (Re 축 코사인 유사도)', fontsize=13)
ax.set_ylabel('B (Im 축 코사인 유사도)', fontsize=13)
ax.set_title('Tri-CHEF Iso-Score 등고선 (s = √(A² + (αB)²), α=0.4)', fontsize=14, pad=14)
ax.grid(True, linestyle='--', alpha=0.4)

legend_handles = [
    mpatches.Patch(color=c, label=f's = {l:.2f}')
    for c, l in zip(contour_colors, levels)
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=10)

plt.tight_layout()
out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig09_isoscore_contour.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

Saved: C:\yssong\KDT-FT-team3-Chainers\DB_insight\Figures\fig09_isoscore_contour.png


C:\Users\sjowu\AppData\Local\Temp\ipykernel_29624\946487025.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# fig09_phase_filter_argand.png - Phase Filter on Argand Plane (Paper Fig.3)

fig, ax = plt.subplots(figsize=(10, 10))

# Unit circle
theta_circle = np.linspace(0, 2 * np.pi, 360)
ax.plot(np.cos(theta_circle), np.sin(theta_circle),
        color='gray', linestyle='--', linewidth=1.2, alpha=0.6, label='단위원')

# Zone boundaries in degrees
trust_deg   = 30
suspect_deg = 80

r_fill = 1.35  # radius used for wedge fills

# Trust zones (upper and lower)
for sign in [1, -1]:
    t_start = 0 if sign == 1 else -trust_deg
    t_end   = trust_deg if sign == 1 else 0
    wedge = Wedge((0, 0), r_fill, t_start, t_end,
                  facecolor='green', alpha=0.18, edgecolor='none')
    ax.add_patch(wedge)

# Suspect zones
for t_start, t_end in [(trust_deg, suspect_deg), (-suspect_deg, -trust_deg)]:
    wedge = Wedge((0, 0), r_fill, t_start, t_end,
                  facecolor='gold', alpha=0.22, edgecolor='none')
    ax.add_patch(wedge)

# Reject zones
for t_start, t_end in [(suspect_deg, 180), (-180, -suspect_deg)]:
    wedge = Wedge((0, 0), r_fill, t_start, t_end,
                  facecolor='red', alpha=0.15, edgecolor='none')
    ax.add_patch(wedge)

# Boundary lines
for deg, style in [(trust_deg, '--'), (suspect_deg, ':')]:
    for sign in [1, -1]:
        rad = np.deg2rad(sign * deg)
        ax.plot([0, r_fill * np.cos(rad)], [0, r_fill * np.sin(rad)],
                color='gray', linestyle=style, linewidth=1.2, alpha=0.7)

# Sample vector z
rho = 0.8
theta_z = np.deg2rad(20)
zx, zy = rho * np.cos(theta_z), rho * np.sin(theta_z)

ax.annotate('', xy=(zx, zy), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='darkblue', lw=2.0))
ax.text(zx + 0.05, zy + 0.05, r'$z = \rho e^{i\theta}$',
        fontsize=13, color='darkblue')

# Angle arc for theta
arc = Arc((0, 0), 0.35, 0.35, angle=0, theta1=0, theta2=20,
          color='darkblue', linewidth=1.5)
ax.add_patch(arc)
ax.text(0.22, 0.06, r'$\theta$', fontsize=13, color='darkblue')

# Rho label along vector
mid_x, mid_y = zx / 2 - 0.07, zy / 2 + 0.05
ax.text(mid_x, mid_y, r'$\rho$', fontsize=13, color='darkblue')

# Zone annotations
ax.text(0.55, 0.12, '신뢰 영역\n(Re·Im 일치)',
        fontsize=11, color='darkgreen', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='honeydew', ec='green', alpha=0.8))
ax.text(0.55, 0.62, '의심 영역',
        fontsize=11, color='goldenrod', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='goldenrod', alpha=0.8))
ax.text(-0.60, 0.60, '거부 영역\n(Re·Im 불일치)',
        fontsize=11, color='darkred', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='mistyrose', ec='red', alpha=0.8))
ax.text(-0.60, -0.60, '거부 영역\n(Re·Im 불일치)',
        fontsize=11, color='darkred', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='mistyrose', ec='red', alpha=0.8))

# Axes
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel('Re (실수부)', fontsize=13)
ax.set_ylabel('Im (허수부)', fontsize=13)
ax.set_title('Phase Filter: Argand 평면 위 신뢰도 게이팅', fontsize=14, pad=14)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.3)

legend_handles = [
    mpatches.Patch(facecolor='green',   alpha=0.4, label=f'신뢰 영역 (|θ| < {trust_deg}°)'),
    mpatches.Patch(facecolor='gold',    alpha=0.5, label=f'의심 영역 ({trust_deg}° ≤ |θ| < {suspect_deg}°)'),
    mpatches.Patch(facecolor='red',     alpha=0.3, label=f'거부 영역 (|θ| ≥ {suspect_deg}°)'),
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=10)

plt.tight_layout()
out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig09_phase_filter_argand.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

Saved: C:\yssong\KDT-FT-team3-Chainers\DB_insight\Figures\fig09_phase_filter_argand.png


C:\Users\sjowu\AppData\Local\Temp\ipykernel_29624\3681072253.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# fig09_miracl_ko_benchmark.png - MIRACL-ko nDCG@10 benchmark (Paper Fig.8)

systems = ['BM25', 'mDPR', 'mContriever', 'mE5-large-v2', 'BGE-M3 dense', 'Tri-CHEF\n(Im axis)']
scores  = [37.10, 41.90, 48.30, 66.50, 69.90, 77.82]

# Sort ascending
order   = np.argsort(scores)
systems = [systems[i] for i in order]
scores  = [scores[i]  for i in order]

bar_colors = ['#C44E52' if 'Tri-CHEF' in s else '#7FB3D8' for s in systems]

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.bar(systems, scores, color=bar_colors, width=0.55, edgecolor='white', linewidth=0.8)

# Value labels on bars
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.7,
            f'{score:.2f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Delta annotation: Tri-CHEF vs BGE-M3
bge_score    = 69.90
trichef_score = 77.82
trichef_idx  = systems.index('Tri-CHEF\n(Im axis)')
bge_idx      = systems.index('BGE-M3 dense')

bge_x    = bars[bge_idx].get_x()    + bars[bge_idx].get_width()    / 2
trichef_x = bars[trichef_idx].get_x() + bars[trichef_idx].get_width() / 2

y_arrow = trichef_score + 4.5
ax.annotate('',
            xy=(trichef_x, trichef_score + 2.0),
            xytext=(bge_x, bge_score + 2.0),
            arrowprops=dict(arrowstyle='->', color='#C44E52', lw=2.0,
                            connectionstyle='arc3,rad=-0.25'))
mid_x = (bge_x + trichef_x) / 2
ax.text(mid_x, y_arrow, '+7.92pp', ha='center', va='bottom',
        fontsize=12, color='#C44E52', fontweight='bold')

# Reference line at 70
ax.axhline(70, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, label='70% 기준선')
ax.text(len(systems) - 0.45, 70.6, '70%', fontsize=10, color='gray')

ax.set_ylim(0, 90)
ax.set_xlabel('검색 시스템', fontsize=13)
ax.set_ylabel('nDCG@10 (%)', fontsize=13)
ax.set_title('MIRACL-ko 벤치마크 성능 비교\nMIRACL-ko dev (213 queries, 1.486M passages)',
             fontsize=14, pad=14)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend(fontsize=10)

plt.tight_layout()
out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig09_miracl_ko_benchmark.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

Saved: C:\yssong\KDT-FT-team3-Chainers\DB_insight\Figures\fig09_miracl_ko_benchmark.png


C:\Users\sjowu\AppData\Local\Temp\ipykernel_29624\373232845.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
